# Tablas finales — esquema estrella / constelación de hechos

Construye las 15 tablas definitivas a partir de los CSV enriquecidos
(`icaa_peliculas.csv`, `icaa_raw.csv`, `direccion.csv`, `guion.csv`,
`creditos_direccion_intermedio.csv`, `creditos_guion_intermedio.csv`,
`distribuidoras.csv`, `peliculas_distribuidoras_temp.csv` -- estas dos
últimas del notebook `3_distribuidoras`), crea las tablas en MySQL con
`FOREIGN KEY` reales, y guarda un CSV por tabla.

**Orden de construcción** (dimensiones antes que uniones, por dependencia de FK):
1. `peliculas` (asigna `pelicula_id`)
2. `distribuidoras` / `peliculas_distribuidoras` (del notebook 3, se remapea `pelicula_id_temp` -> `pelicula_id`)
3. `generos` / `peliculas_generos` / `peliculas_generos_padre`
4. `paises` / `participacion_pais`
5. `empresas_productoras` / `peliculas_empresas_productoras`
6. `direccion` / `guion` (remapeo de texto a entero) → `peliculas_direccion` / `peliculas_guion`
7. `taquilla_cine_esp` (requiere `icaa_raw.csv`)
8. `subvenciones`
9. Creación de tablas en MySQL (DDL con FK) + inserción + CSV
10. Vista `vista_latam`

## 1. Librerías y carga de datos

In [15]:
import os
import re
import json
import csv
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv()

BASE = Path("..")
CSV_DIR = BASE / "3 - csv"

icaa_peliculas = pd.read_csv(CSV_DIR / "icaa_peliculas.csv", sep=';')
icaa_raw = pd.read_csv(CSV_DIR / "icaa_raw.csv", sep=';')
direccion = pd.read_csv(CSV_DIR / "direccion.csv", sep=';')
guion = pd.read_csv(CSV_DIR / "guion.csv", sep=';')
creditos_direccion_intermedio = pd.read_csv(CSV_DIR / "creditos_direccion_intermedio.csv", sep=';')
creditos_guion_intermedio = pd.read_csv(CSV_DIR / "creditos_guion_intermedio.csv", sep=';')

print(f"icaa_peliculas: {len(icaa_peliculas)}")
print(f"icaa_raw: {len(icaa_raw)}")
print(f"direccion: {len(direccion)} | guion: {len(guion)}")
print(f"creditos_direccion_intermedio: {len(creditos_direccion_intermedio)} | "
      f"creditos_guion_intermedio: {len(creditos_guion_intermedio)}")


icaa_peliculas: 2052
icaa_raw: 2812
direccion: 1649 | guion: 2323
creditos_direccion_intermedio: 2159 | creditos_guion_intermedio: 2992


### Corregir `icaa_raw` para el caso de "Cant dels ocells, El"

El título en `icaa_raw` sigue siendo el original sin separar
(`"... Sagrera"`) -- se corrige aquí para que la unión de la sección
`taquilla_cine_esp` la encuentre (si no, se queda como la única fila sin
`pelicula_id` tras la unión).

In [16]:
mascara_raw = (icaa_raw['titulo'] == 'Cant dels ocells, El Sagrera') & (icaa_raw['fecha_estreno'] == '19/12/2008')
icaa_raw.loc[mascara_raw, 'titulo'] = 'Cant dels ocells, El'
print(f"Filas corregidas en icaa_raw: {mascara_raw.sum()}")


Filas corregidas en icaa_raw: 1


## 2. `peliculas`

In [17]:
peliculas_completo = icaa_peliculas.copy()
peliculas_completo.insert(0, 'pelicula_id', range(1, len(peliculas_completo) + 1))
peliculas_completo = peliculas_completo.rename(columns={'distribuidora_final': 'distribuidora'})

ambos = peliculas_completo[peliculas_completo['icaa_id'].notna() & peliculas_completo['tmdb_id'].notna()]
assert len(ambos) == 0, f"{len(ambos)} filas con icaa_id Y tmdb_id a la vez -- revisar"

# peliculas_completo se conserva con TODAS las columnas (se usa más abajo para
# extraer generos, nacionalidad, empresas productoras, subvenciones); "peliculas"
# es la versión recortada que se sube tal cual a MySQL/CSV. NOTA: ya NO incluye
# distribuidora/distribuidora_fuente -- eso pasó a distribuidoras/peliculas_distribuidoras.
peliculas = peliculas_completo[[
    'pelicula_id', 'icaa_id', 'tmdb_id', 'titulo', 'titulo_icaa', 'fecha_estreno', 'anio',
    'recaudacion_total', 'espectadores_total', 'subvenciones_total',
]]

print(f"✓ peliculas: {len(peliculas)} filas, pelicula_id de 1 a {peliculas['pelicula_id'].max()}")
peliculas.head()


✓ peliculas: 2052 filas, pelicula_id de 1 a 2052


,pelicula_id,icaa_id,tmdb_id,titulo,titulo_icaa,fecha_estreno,anio,recaudacion_total,espectadores_total,subvenciones_total
0,1,NaN,NaN,"091, policia al habla (1960) (re)",NaN,16/05/2025,2025,42.0,12,NaN
1,2,100317.0,NaN,Todos lo saben,TODOS LO SABEN,14/09/2018,2022,83.0,18,280000.0
2,3,100421.0,NaN,"Generacion silenciosa, La",LA GENERACIÓN SILENCIOSA,10/09/2021,2021,129.0,35,0.0
3,4,100517.0,NaN,Platon,PLATÓN,08/10/2021,2021,237.0,60,0.0
4,5,100820.0,NaN,Agua,AGUA,11/02/2022,2022,2296.0,357,0.0


## 2b. `distribuidoras` / `peliculas_distribuidoras`

Ya construidas en `3_distribuidoras.ipynb`. Solo hace falta remapear
`pelicula_id_temp` (posición en `icaa_peliculas.csv`, base 0) a `pelicula_id`
(base 1, asignado arriba) -- como ambos se derivan del mismo orden de filas
de `icaa_peliculas.csv` sin reordenar ni filtrar, `pelicula_id = pelicula_id_temp + 1`
es una correspondencia exacta.

In [18]:
distribuidoras = pd.read_csv(CSV_DIR / "distribuidoras.csv", sep=';')
peliculas_distribuidoras_temp = pd.read_csv(CSV_DIR / "peliculas_distribuidoras_temp.csv", sep=';')

peliculas_distribuidoras_temp['pelicula_id'] = peliculas_distribuidoras_temp['pelicula_id_temp'] + 1
peliculas_distribuidoras = peliculas_distribuidoras_temp[['pelicula_id', 'distribuidora_id', 'fuente']]

# Verificación: el pelicula_id más alto no debe superar el total de películas
assert peliculas_distribuidoras['pelicula_id'].max() <= len(peliculas), \
    "pelicula_id fuera de rango -- revisar que icaa_peliculas.csv no cambió entre notebooks"

print(f"✓ distribuidoras: {len(distribuidoras)} distribuidoras únicas")
print(f"✓ peliculas_distribuidoras: {len(peliculas_distribuidoras)} filas, "
      f"{peliculas_distribuidoras['pelicula_id'].nunique()} películas cubiertas")


✓ distribuidoras: 459 distribuidoras únicas
✓ peliculas_distribuidoras: 2165 filas, 2052 películas cubiertas


## 3. `generos` / `peliculas_generos`

Regla acordada: los géneros de `ficha_genero` van primero y en su orden
original, salvo `Drama` -- si aparece junto a otro género, se manda al
final (para que tomar solo el primer género de una película no caiga
sistemáticamente en el genérico "Drama"). El género de `ficha_tipo`
(respaldo cuando no hay género real) siempre va después, y se descarta el
prefijo de formato ("Cine"/"Video"), quedándonos solo con la parte real
(ej. "Ficción").

In [19]:
ALIAS_GENERO = {
    'Accion': 'Acción',  # ICAA escribió "ACCION" sin tilde en al menos un caso
}


def ordenar_generos_pelicula(genero_final, genero_fuente):
    """
    ICAA usa separadores inconsistentes para valores compuestos: " - " normal,
    "\u2013" (guion largo, a veces sin espacio), o "-" pegado sin espacios con
    2-3 géneros seguidos ("Aventuras-Misterio-Familiar",
    "THRILLER-DRAMA-ACCION"). El regex cubre los tres. ALIAS_GENERO corrige
    casos conocidos de tilde perdida en mayúsculas para que no generen una
    entrada de género duplicada.
    """
    if pd.isna(genero_final) or pd.isna(genero_fuente):
        return []
    partes = [p.strip().capitalize() for p in re.split(r'\s*[-\u2013\u2014]\s*', genero_final) if p.strip()]
    partes = [ALIAS_GENERO.get(p, p) for p in partes]
    if genero_fuente == 'ficha_genero':
        if 'Drama' in partes and len(partes) > 1:
            otros = [g for g in partes if g != 'Drama']
            ordenados = otros + ['Drama']
        else:
            ordenados = partes
        return [(g, i, 'ficha_genero') for i, g in enumerate(ordenados)]
    elif genero_fuente == 'ficha_tipo':
        return [(partes[-1], 0, 'ficha_tipo')]
    return []


registros_generos = []
for _, row in peliculas_completo.iterrows():
    for genero, orden, fuente in ordenar_generos_pelicula(row['genero_final'], row['genero_fuente']):
        registros_generos.append({
            'pelicula_id': row['pelicula_id'], 'genero_nombre': genero,
            'orden': orden, 'fuente': fuente,
        })

peliculas_generos_temp = pd.DataFrame(registros_generos)
print(f"✓ {len(peliculas_generos_temp)} relaciones película-género extraídas")
peliculas_generos_temp.head()


✓ 1871 relaciones película-género extraídas


,pelicula_id,genero_nombre,orden,fuente
0,2,Drama,0,ficha_genero
1,3,Comedia,0,ficha_genero
2,3,Drama,1,ficha_genero
3,4,Ficción,0,ficha_genero
4,5,Drama,0,ficha_genero


In [20]:
generos_unicos = sorted(peliculas_generos_temp['genero_nombre'].unique())
generos = pd.DataFrame({
    'genero_id': range(1, len(generos_unicos) + 1),
    'nombre': generos_unicos,
})

peliculas_generos = peliculas_generos_temp.merge(
    generos, left_on='genero_nombre', right_on='nombre', how='left', validate='many_to_one'
)[['pelicula_id', 'genero_id', 'orden', 'fuente']]

print(f"✓ generos: {len(generos)} géneros únicos")
print(f"✓ peliculas_generos: {len(peliculas_generos)} filas")
generos.head(10)


✓ generos: 69 géneros únicos
✓ peliculas_generos: 1871 filas


,genero_id,nombre
0,1,Acción
1,2,Adolescentes
2,3,Animación
3,4,Animación 3d
4,5,Arte
5,6,Aventuras
6,7,Biográfica
7,8,Ciencia ficción
8,9,Comedia
9,10,Comedia de acción


### Diagnóstico -- géneros en `orden=0` por película

Cuántas películas tiene cada género cuando se toma solo el primero (el más
específico, según la regla acordada: `Drama` siempre queda relegado si
acompaña a otro).

In [21]:
primeros = peliculas_generos[peliculas_generos['orden'] == 0].merge(generos, on='genero_id')

print(f"Total con género en orden 0: {len(primeros)} / {len(peliculas)} películas")
print()
print(primeros['nombre'].value_counts().to_string())


Total con género en orden 0: 1775 / 2052 películas

nombre
Documental                   463
Drama                        366
Comedia                      210
Ficción                      115
Thriller                      77
Documental social             44
Animación                     40
Biográfica                    30
Terror                        28
Experimental                  25
Documental biográfico         23
Documental musical            20
Documental artes              19
Drama social                  19
Aventuras                     19
Acción                        16
Docuficción                   15
Musical                       15
Drama histórico               13
Comedia dramática             13
Histórica                     13
Comedia romántica             13
Thriller psicológico          13
Drama romántica               12
Suspense                      10
Fantasía                       9
Ciencia ficción                9
Thriller drama                 9
Comedia negra    

### `peliculas_generos_padre` -- agrupación temática

Colapsa subgéneros afines a una categoría más amplia, para análisis
agregado (crosstabs, gráficos) sin perder el detalle fino de `generos`.
Reglas acordadas:
- Prefijo `Documental` (y `Docuficción`, `Docudrama`) → `Documental`
- Prefijo `Animación` (ej. `Animación 3d`) → `Animación` -- no es un
  género en sentido estricto, pero se acepta como categoría padre
- `Falso documental` → `Ficción` (excepción)
- `Misterio`, `Negro`, `Crimen`, `Policíaca` → `Thriller`
- Cualquier género que contenga "drama" → también cuenta como `Drama`
  (incluye `Melodrama`; NO usa comparación sin tildes, así que
  `"dramática"` no activa esta regla por el acento)
- Cualquier género que contenga "comedia" → también cuenta como `Comedia`;
  si además lleva " de " (ej. `"Comedia de acción"`), se añade un segundo
  padre con lo que sigue a "de" (`Acción`)
- Cualquier género que contenga "thriller" → también cuenta como `Thriller`
- `Tragicomedia` se queda como su propio género -- EXCEPTO
  `"Comedia dramática"` (que sí es distinta de `Tragicomedia`), que solo
  se convierte en `Tragicomedia` cuando viene de `ficha_genero`; si viniera
  de `ficha_tipo` (nunca ha pasado en este corpus, pero por si acaso) se
  reparte en `Comedia` + `Drama`

Como un género puede resolver a más de un padre a la vez (ej. "Comedia de
acción" cuenta para `Comedia` Y `Acción`), esto es una tabla propia
(pelicula_id, genero_padre), no una columna en `generos`.

In [22]:
GENEROS_A_THRILLER = {'misterio', 'negro', 'crimen', 'policíaca'}


def resolver_genero_padre(nombre, fuente):
    n = nombre.lower()

    if n in ('comedia dramatica', 'comedia dramática'):
        return ['Tragicomedia'] if fuente == 'ficha_genero' else ['Comedia', 'Drama']
    if n == 'tragicomedia':
        return ['Tragicomedia']
    if n == 'docudrama':
        return ['Documental']
    if n == 'falso documental':
        return ['Ficción']
    if n.startswith('documental') or n == 'docuficción':
        return ['Documental']
    if n.startswith('animación') or n.startswith('animacion'):
        return ['Animación']
    if n in GENEROS_A_THRILLER:
        return ['Thriller']

    # Orden de PRIORIDAD (no alfabético): Comedia y Thriller van primero si
    # aparecen, para que al quedarnos con el primer padre el resultado sea
    # el esperado ("Comedia de acción" -> Comedia, "Thriller drama" -> Thriller).
    padres = []
    if 'comedia' in n:
        padres.append('Comedia')
        if ' de ' in n:
            resto = n.split(' de ', 1)[1].strip()
            padres.append(resto.capitalize())
    if 'thriller' in n and 'Thriller' not in padres:
        padres.append('Thriller')
    if 'drama' in n and 'Drama' not in padres:
        padres.append('Drama')

    return padres if padres else [nombre]


# Se parte SOLO del género en orden=0 (el más específico por película), y de
# los posibles padres resueltos nos quedamos solo con el primero (prioridad),
# para que el resultado sea exactamente una fila por película, sin duplicados.
generos_orden0 = peliculas_generos[peliculas_generos['orden'] == 0].merge(generos, on='genero_id')

registros_padre = []
for _, row in generos_orden0.iterrows():
    padres_resueltos = resolver_genero_padre(row['nombre'], row['fuente'])
    registros_padre.append({'pelicula_id': row['pelicula_id'], 'genero_padre': padres_resueltos[0]})

peliculas_generos_padre = pd.DataFrame(registros_padre)

duplicados = peliculas_generos_padre['pelicula_id'].duplicated().sum()
assert duplicados == 0, f"{duplicados} películas con más de un genero_padre -- no debería pasar"

print(f"✓ peliculas_generos_padre: {len(peliculas_generos_padre)} filas (1 por película), "
      f"{peliculas_generos_padre['genero_padre'].nunique()} categorías padre")
print()
print(peliculas_generos_padre['genero_padre'].value_counts().head(20))


✓ peliculas_generos_padre: 1775 filas (1 por película), 37 categorías padre

genero_padre
Documental           613
Drama                423
Comedia              242
Ficción              116
Thriller             102
Animación             42
Biográfica            30
Terror                28
Experimental          25
Aventuras             19
Tragicomedia          17
Acción                16
Musical               15
Histórica             13
Suspense              10
Fantasía               9
Ciencia ficción        9
Arte                   7
Terror fantastico      5
Creación               4
Name: count, dtype: int64


## 4. `paises` / `participacion_pais`

Fuente principal: `nacionalidad_paises_icaa` (`"España (50.0%); Francia (40.0%)"`).
Si una película no tiene ese dato pero sí `empresas_productoras_icaa` (con
desglose de país por empresa), se propaga el porcentaje desde ahí --
`porcentaje_fuente` distingue de dónde vino cada fila.

In [23]:
PAISES_LATAM_NOMBRES = {
    'México', 'Argentina', 'Brasil', 'Chile', 'Colombia', 'Perú', 'Venezuela',
    'Uruguay', 'Paraguay', 'Bolivia', 'Ecuador', 'Costa Rica', 'Panamá',
    'Guatemala', 'Honduras', 'El Salvador', 'Nicaragua', 'República Dominicana',
    'Cuba', 'Puerto Rico',
}


def parsear_nacionalidad(nacionalidad_str):
    if pd.isna(nacionalidad_str) or not nacionalidad_str:
        return []
    resultado = []
    for parte in nacionalidad_str.split(';'):
        m = re.match(r'^(.+?)\s*\(([\d.]+)%\)\s*$', parte.strip())
        if m:
            resultado.append((m.group(1).strip(), float(m.group(2))))
    return resultado


def parsear_paises_desde_empresas(empresas_json_str):
    """
    Deduplica por país (cada empresa del mismo país repite el mismo
    porcentaje_pais), para usar como respaldo cuando no hay nacionalidad_paises_icaa.
    """
    if pd.isna(empresas_json_str) or not empresas_json_str:
        return []
    try:
        empresas = json.loads(empresas_json_str)
    except Exception:
        return []
    vistos = {}
    for e in empresas:
        pais = e.get('pais', '').strip().title()
        pct = e.get('porcentaje_pais')
        if pais and pct is not None and pais not in vistos:
            vistos[pais] = pct
    return list(vistos.items())


registros_pais = []
for _, row in peliculas_completo.iterrows():
    desde_nacionalidad = parsear_nacionalidad(row['nacionalidad_paises_icaa'])
    if desde_nacionalidad:
        for pais, pct in desde_nacionalidad:
            registros_pais.append({
                'pelicula_id': row['pelicula_id'], 'pais_nombre': pais,
                'porcentaje': pct, 'porcentaje_fuente': 'nacionalidad_icaa',
            })
        continue

    desde_empresas = parsear_paises_desde_empresas(row['empresas_productoras_icaa'])
    for pais, pct in desde_empresas:
        registros_pais.append({
            'pelicula_id': row['pelicula_id'], 'pais_nombre': pais,
            'porcentaje': pct, 'porcentaje_fuente': 'empresas_productoras_icaa',
        })

participacion_pais_temp = pd.DataFrame(registros_pais)
print(f"✓ {len(participacion_pais_temp)} relaciones película-país extraídas")
print(participacion_pais_temp['porcentaje_fuente'].value_counts())


✓ 2247 relaciones película-país extraídas
porcentaje_fuente
nacionalidad_icaa            2246
empresas_productoras_icaa       1
Name: count, dtype: int64


In [24]:
paises_unicos = sorted(participacion_pais_temp['pais_nombre'].unique())
paises = pd.DataFrame({
    'pais_id': range(1, len(paises_unicos) + 1),
    'nombre': paises_unicos,
})
paises['es_latam'] = paises['nombre'].isin(PAISES_LATAM_NOMBRES)

participacion_pais = participacion_pais_temp.merge(
    paises, left_on='pais_nombre', right_on='nombre', how='left', validate='many_to_one'
)[['pelicula_id', 'pais_id', 'porcentaje', 'porcentaje_fuente']]

print(f"✓ paises: {len(paises)} países únicos, {paises['es_latam'].sum()} marcados LATAM")
print(f"✓ participacion_pais: {len(participacion_pais)} filas")
paises[paises['es_latam']]


✓ paises: 55 países únicos, 15 marcados LATAM
✓ participacion_pais: 2247 filas


,pais_id,nombre,es_latam
2,3,Argentina,True
5,6,Brasil,True
9,10,Chile,True
11,12,Colombia,True
12,13,Costa Rica,True
13,14,Cuba,True
15,16,Ecuador,True
32,33,México,True
33,34,Nicaragua,True
35,36,Panamá,True


## 5. `empresas_productoras` / `peliculas_empresas_productoras`

Mismo país normalizado (`.title()`) para que coincida con `paises` sin
crear entradas duplicadas por diferencia de mayúsculas
(`"ESPAÑA"` en este campo vs `"España"` en `nacionalidad_paises_icaa`).

In [25]:
def parsear_empresas(empresas_json_str):
    if pd.isna(empresas_json_str) or not empresas_json_str:
        return []
    try:
        return json.loads(empresas_json_str)
    except Exception:
        return []


def recuperar_porcentaje_embebido(nombre_crudo, porcentaje_actual):
    """
    Bug de parseo en notebook 1: en algunos casos el porcentaje de la
    empresa quedó pegado al nombre en vez de extraerse a su propio campo
    (ej. "100 BARES (1%)" con porcentaje_empresa=None). Se recupera aquí
    cuando el campo real está vacío.
    """
    m = re.search(r'\((\d+(?:\.\d+)?)\s*%\)\s*$', nombre_crudo)
    if not m:
        return nombre_crudo, porcentaje_actual
    nombre_limpio = nombre_crudo[:m.start()].strip()
    if porcentaje_actual is None:
        return nombre_limpio, float(m.group(1))
    return nombre_limpio, porcentaje_actual


registros_empresas = []
for _, row in peliculas_completo.iterrows():
    for e in parsear_empresas(row['empresas_productoras_icaa']):
        nombre_crudo = e.get('empresa', '').strip()
        nombre_limpio, porcentaje = recuperar_porcentaje_embebido(nombre_crudo, e.get('porcentaje_empresa'))
        registros_empresas.append({
            'pelicula_id': row['pelicula_id'],
            'pais_nombre': e.get('pais', '').strip().title(),
            'empresa_nombre': nombre_limpio,
            'porcentaje_empresa': porcentaje,
        })

peliculas_empresas_temp = pd.DataFrame(registros_empresas)
print(f"✓ {len(peliculas_empresas_temp)} relaciones película-empresa extraídas")


✓ 3985 relaciones película-empresa extraídas


In [26]:
import unicodedata

RAZONES_SOCIALES = {
    'S','SL','SA','SAU','SLU','SRL','SARL','SAS','SASU','SAPI','LDA','LTD','LTDA',
    'LIMITADA','INC','GMBH','BV','BVBA','AG','KG','LLC','LLP','PTY','OOD','SPRL',
    'AIE','SCOOP','EIRL','CV','RL','PC','KFT','AB','AS','ASA','SPA','APS','OY',
}
CONECTORES_LEGAL = {'DE', 'DEL', 'Y', 'UNIPERSONALE', 'UNIPESSOAL'}
GENERICAS = {'FILMS', 'FILM', 'PICTURES', 'PRODUCCIONES', 'PRODUCTIONS', 'PRODUCTION', 'CINE'}


def quitar_acentos(s):
    return ''.join(c for c in unicodedata.normalize('NFKD', s) if not unicodedata.combining(c))


def normalizar_token(tok):
    return tok.replace('.', '').replace(',', '')


def limpiar_nombre_empresa(nombre, quitar_tildes, quitar_genericas):
    """
    Quita razón social (lista explícita de formas legales -- un heurístico
    por longitud resultó inseguro, se probó y recortaba nombres reales como
    "BD CINE" -> "BD") y el sufijo "- Financiera". `quitar_genericas` además
    quita palabras descriptivas (Films/Producciones/...) -- solo se usa para
    detectar duplicados y nombrar los grupos que sí lo son, nunca para
    renombrar una empresa que aparece una sola vez.
    """
    n = quitar_acentos(nombre.upper()) if quitar_tildes else nombre.upper()
    tokens = n.split()
    if len(tokens) >= 2 and tokens[-1] == 'FINANCIERA' and tokens[-2] in ('-', '–', '—'):
        tokens = tokens[:-2]
    while len(tokens) > 1:
        ultimo = normalizar_token(tokens[-1])
        if ultimo in RAZONES_SOCIALES or tokens[-1] in CONECTORES_LEGAL:
            tokens.pop()
        elif quitar_genericas and ultimo in GENERICAS:
            tokens.pop()
        else:
            break
    return ' '.join(tokens).rstrip(',').strip()


def elegir_nombre_canonico(variantes, con_genericas):
    """Entre las variantes de un grupo, prioriza la que tenga tildes (más
    fiel al nombre real) sobre la que no."""
    candidatos = [limpiar_nombre_empresa(v, False, con_genericas) for v in variantes]
    con_tildes = [c for c in candidatos if c != quitar_acentos(c)]
    elegido = con_tildes[0] if con_tildes else candidatos[0]
    palabras = elegido.split()
    conectores_minuscula = {'de', 'del', 'la', 'las', 'el', 'los', 'y'}
    return ' '.join(w.lower() if w.lower() in conectores_minuscula else w.capitalize() for w in palabras)


nombres_unicos_crudos = peliculas_empresas_temp['empresa_nombre'].unique()

# Clave de agrupación: razón social + genéricas fuera, para detectar duplicados
clave_grupo = {n: limpiar_nombre_empresa(n, True, True) for n in nombres_unicos_crudos}
grupos = {}
for nombre, clave in clave_grupo.items():
    grupos.setdefault(clave, []).append(nombre)

# Nombre final por cada nombre crudo: si su grupo tiene más de una variante,
# usa el canónico (con genéricas fuera); si aparece solo, solo se le quita
# la razón social (nunca las palabras genéricas, para no arriesgar recortar
# una marca real que nunca tuvo que desambiguarse de nada).
nombre_final_por_crudo = {}
for clave, variantes in grupos.items():
    if len(variantes) > 1:
        canonico = elegir_nombre_canonico(variantes, con_genericas=True)
        for v in variantes:
            nombre_final_por_crudo[v] = canonico
    else:
        nombre_final_por_crudo[variantes[0]] = elegir_nombre_canonico(variantes, con_genericas=False)

peliculas_empresas_temp['empresa_nombre_final'] = peliculas_empresas_temp['empresa_nombre'].map(nombre_final_por_crudo)

grupos_fusionados = sum(1 for v in grupos.values() if len(v) > 1)
print(f"✓ {len(nombres_unicos_crudos)} nombres crudos -> {len(set(nombre_final_por_crudo.values()))} empresas únicas "
      f"tras normalizar ({grupos_fusionados} grupos fusionados)")

empresas_unicas = sorted(set(nombre_final_por_crudo.values()))
empresas_productoras = pd.DataFrame({
    'empresa_id': range(1, len(empresas_unicas) + 1),
    'nombre': empresas_unicas,
})

peliculas_empresas_productoras = (
    peliculas_empresas_temp
    .merge(empresas_productoras, left_on='empresa_nombre_final', right_on='nombre', how='left', validate='many_to_one')
    .merge(paises, left_on='pais_nombre', right_on='nombre', how='left', suffixes=('', '_pais'))
)[['pelicula_id', 'empresa_id', 'pais_id', 'porcentaje_empresa']]

sin_pais = peliculas_empresas_productoras['pais_id'].isna().sum()
print(f"✓ empresas_productoras: {len(empresas_productoras)} empresas únicas")
print(f"✓ peliculas_empresas_productoras: {len(peliculas_empresas_productoras)} filas "
      f"({sin_pais} sin pais_id resuelto -- revisar si es alto)")


✓ 2344 nombres crudos -> 2279 empresas únicas tras normalizar (44 grupos fusionados)
✓ empresas_productoras: 2279 empresas únicas
✓ peliculas_empresas_productoras: 3985 filas (1 sin pais_id resuelto -- revisar si es alto)


## 6. `direccion` / `guion` -- remapeo a clave entera

`dir_id`/`gui_id` venían como texto (`"dir_0001"`) del notebook de
enriquecimiento. Se sustituyen por enteros reales, guardando el mapa de
conversión para aplicarlo también a las tablas de créditos intermedias.

In [27]:
def remapear_id_texto_a_entero(dim_df, id_col_texto, id_col_entero_nuevo, prefijo):
    dim_df = dim_df.copy()
    if id_col_texto in dim_df.columns:
        mapa = {id_texto: i + 1 for i, id_texto in enumerate(dim_df[id_col_texto])}
        dim_df.insert(0, id_col_entero_nuevo, dim_df[id_col_texto].map(mapa))
        dim_df = dim_df.drop(columns=[id_col_texto])
    else:
        # Este notebook sobrescribe direccion.csv/guion.csv en su propia sección
        # de exportación (13) -- si ya se corrió una vez, dir_id/gui_id ya no
        # existe en el CSV, solo director_id/guionista_id. Se reconstruye el
        # mapa con la misma convención de nombres de notebook 2 en vez de fallar.
        assert id_col_entero_nuevo in dim_df.columns, \
            f"Ni {id_col_texto} ni {id_col_entero_nuevo} están en el CSV -- algo más está mal"
        mapa = {f"{prefijo}_{i:04d}": i for i in dim_df[id_col_entero_nuevo]}
    return dim_df, mapa


direccion, mapa_dir = remapear_id_texto_a_entero(direccion, 'dir_id', 'director_id', 'dir')
guion, mapa_gui = remapear_id_texto_a_entero(guion, 'gui_id', 'guionista_id', 'gui')

if 'person_id' in direccion.columns:
    direccion = direccion.rename(columns={'person_id': 'person_id_tmdb'})
if 'person_id' in guion.columns:
    guion = guion.rename(columns={'person_id': 'person_id_tmdb'})

print(f"✓ direccion: {len(direccion)} filas, director_id de 1 a {direccion['director_id'].max()}")
print(f"✓ guion: {len(guion)} filas, guionista_id de 1 a {guion['guionista_id'].max()}")
direccion.head(3)


✓ direccion: 1649 filas, director_id de 1 a 1649
✓ guion: 2323 filas, guionista_id de 1 a 2323


,director_id,person_id_tmdb,nombre,sexo,fuente_sexo,pais,fuente_pais,nombre_credito_original
0,1,229931.0,Asghar Farhadi,hombre,tmdb,IR,tmdb_heuristico,NaN
1,2,1930872.0,Ferrán Navarro-Beltrán,hombre,tmdb,NaN,NaN,NaN
2,3,1114075.0,Vicente Pérez Herrero,NaN,NaN,NaN,NaN,NaN


## 7. `peliculas_direccion` / `peliculas_guion`

Une los créditos intermedios (`icaa_id`/`tmdb_id` + `dir_id`/`gui_id`) con
`peliculas` (para obtener `pelicula_id`) y con el mapa de remapeo (para
obtener `director_id`/`guionista_id`).

### Diagnóstico -- créditos huérfanos (sin `dir_id`/`gui_id`)

Puede pasar si una corrección manual sobre `direccion`/`guion` (como la de
"Juan Prosper") se aplicó DESPUÉS de exportar `creditos_..._intermedio.csv`
-- ese crédito quedó sin id asignado en el momento del cálculo. Sin este
filtro, `guionista_id`/`director_id` llegarían como `NaN` y romperían el
`INSERT` en MySQL (la clave primaria no admite `NULL`).

`creditos_..._intermedio` no guarda el nombre de la persona (solo
`icaa_id`/`tmdb_id`), así que no se puede reparar automáticamente por
nombre -- se corrige a mano el caso ya identificado (Juan Prosper,
`icaa_id=573651`, guionista) y cualquier otro huérfano que aparezca se
reporta y se descarta, documentado, en vez de romper el `INSERT` en
silencio.

In [28]:
# Caso conocido: "Juan Prosper" (El vuelo de la cigüeña, icaa_id=573651) --
# la corrección manual sobre `guion` se hizo después de exportar
# creditos_guion_intermedio.csv, así que este crédito quedó sin gui_id.
CORRECCIONES_CREDITOS_HUERFANOS = {
    ('guion', 573651.0): 'gui_1440',
}


def reparar_o_descartar_huerfanos(creditos_df, id_col, mapa, rol):
    huerfanos = creditos_df[creditos_df[id_col].isna()]
    if len(huerfanos) == 0:
        print(f"✓ Sin huérfanos en {id_col}")
        return creditos_df

    print(f"⚠ {len(huerfanos)} créditos sin {id_col} tras el mapeo")
    reparados, sin_reparar = [], []

    for idx_h, fila in huerfanos.iterrows():
        clave = (rol, fila['icaa_id'])
        id_texto = CORRECCIONES_CREDITOS_HUERFANOS.get(clave)
        if id_texto and id_texto in mapa:
            creditos_df.loc[idx_h, id_col] = mapa[id_texto]
            reparados.append(idx_h)
        else:
            sin_reparar.append(idx_h)

    print(f"  Reparados manualmente: {len(reparados)} / {len(huerfanos)}")
    if sin_reparar:
        print(f"  ⚠ Sin reparar -- se descartan (revisar si hace falta añadirlos "
              f"a CORRECCIONES_CREDITOS_HUERFANOS): {len(sin_reparar)}")
        print(creditos_df.loc[sin_reparar])
        creditos_df = creditos_df.drop(index=sin_reparar)

    return creditos_df


creditos_direccion_intermedio['director_id'] = creditos_direccion_intermedio['dir_id'].map(mapa_dir)
creditos_guion_intermedio['guionista_id'] = creditos_guion_intermedio['gui_id'].map(mapa_gui)

creditos_direccion_intermedio = reparar_o_descartar_huerfanos(
    creditos_direccion_intermedio, 'director_id', mapa_dir, 'direccion'
)
creditos_guion_intermedio = reparar_o_descartar_huerfanos(
    creditos_guion_intermedio, 'guionista_id', mapa_gui, 'guion'
)


✓ Sin huérfanos en director_id
⚠ 1 créditos sin guionista_id tras el mapeo
  Reparados manualmente: 1 / 1


In [29]:
def clave_identificacion(icaa_id, tmdb_id):
    """
    Clave compuesta que identifica una película sin ambigüedad, evitando el
    problema de que pandas SÍ empareja NaN==NaN en un merge -- con dos merges
    sucesivos (uno por icaa_id, otro por tmdb_id), todas las filas con
    tmdb_id vacío se emparejaban entre sí por error (explosión cartesiana:
    3.9M filas en vez de ~2200). Con una única clave compuesta, nunca hay
    NaN de por medio en el merge real.
    """
    if pd.notna(icaa_id):
        return f"icaa_{int(icaa_id)}"
    elif pd.notna(tmdb_id):
        return f"tmdb_{int(tmdb_id)}"
    return None


peliculas['clave_pelicula'] = peliculas.apply(
    lambda r: clave_identificacion(r['icaa_id'], r['tmdb_id']), axis=1
)


def unir_con_pelicula_id(creditos_df):
    creditos_df = creditos_df.copy()
    creditos_df['clave_pelicula'] = creditos_df.apply(
        lambda r: clave_identificacion(r['icaa_id'], r['tmdb_id']), axis=1
    )
    creditos_df = creditos_df.merge(
        peliculas[['pelicula_id', 'clave_pelicula']], on='clave_pelicula', how='left'
    )
    return creditos_df.drop(columns=['clave_pelicula'])



peliculas_direccion = unir_con_pelicula_id(creditos_direccion_intermedio)[
    ['pelicula_id', 'director_id', 'fuente_credito']
]
peliculas_guion = unir_con_pelicula_id(creditos_guion_intermedio)[
    ['pelicula_id', 'guionista_id', 'fuente_credito']
]

sin_pelicula_dir = peliculas_direccion['pelicula_id'].isna().sum()
sin_pelicula_gui = peliculas_guion['pelicula_id'].isna().sum()
print(f"✓ peliculas_direccion: {len(peliculas_direccion)} filas ({sin_pelicula_dir} sin pelicula_id -- revisar si es alto)")
print(f"✓ peliculas_guion: {len(peliculas_guion)} filas ({sin_pelicula_gui} sin pelicula_id -- revisar si es alto)")

peliculas_direccion = peliculas_direccion.dropna(subset=['pelicula_id'])
peliculas_guion = peliculas_guion.dropna(subset=['pelicula_id'])
peliculas_direccion['pelicula_id'] = peliculas_direccion['pelicula_id'].astype(int)
peliculas_guion['pelicula_id'] = peliculas_guion['pelicula_id'].astype(int)

# 'clave_pelicula' era solo un auxiliar interno para el merge -- no debe
# persistir en 'peliculas' (ni a MySQL ni al CSV), así que se descarta aquí,
# una vez que ya no hace falta.
peliculas = peliculas.drop(columns=['clave_pelicula'])


✓ peliculas_direccion: 2159 filas (0 sin pelicula_id -- revisar si es alto)
✓ peliculas_guion: 2992 filas (0 sin pelicula_id -- revisar si es alto)


## 8. `taquilla_cine_esp`

`icaa_raw.csv` trae una fila por año en cartelera. La unión se hace en dos
pasos: **primero por `icaa_id`** (para las filas que lo tienen -- incluye
reposiciones/reestrenos, que comparten `icaa_id` con su estreno original
pero tienen un `(título, fecha_estreno)` propio que no sobrevivió a la
agrupación de notebook 1), y **por `(título, fecha_estreno)` solo para las
que no tienen `icaa_id`** (las ~277 sin ficha ICAA). Unir todo por título
desde el principio dejaba 36 filas sin emparejar; por `icaa_id` primero,
solo queda 1 (la de "Sagrera", ya conocida -- ver la corrección manual que
ya aplicaste sobre `icaa_peliculas` antes de este notebook).

In [30]:
con_icaa_id = icaa_raw[icaa_raw['icaa_id'].notna()].merge(
    peliculas.loc[peliculas['icaa_id'].notna(), ['pelicula_id', 'icaa_id']],
    on='icaa_id', how='left', validate='many_to_one'
)
sin_icaa_id = icaa_raw[icaa_raw['icaa_id'].isna()].merge(
    peliculas[['pelicula_id', 'titulo', 'fecha_estreno']],
    on=['titulo', 'fecha_estreno'], how='left'
)
icaa_raw_con_pelicula = pd.concat([con_icaa_id, sin_icaa_id], ignore_index=True)

sin_match = icaa_raw_con_pelicula[icaa_raw_con_pelicula['pelicula_id'].isna()]
print(f"Filas de icaa_raw sin pelicula_id tras la unión: {len(sin_match)} / {len(icaa_raw_con_pelicula)}")
if len(sin_match):
    print(sin_match[['titulo', 'fecha_estreno', 'icaa_id']].to_string())

taquilla_cine_esp = (
    icaa_raw_con_pelicula.dropna(subset=['pelicula_id'])
    .groupby(['pelicula_id', 'anio'], as_index=False)
    .agg(recaudacion=('recaudacion', 'sum'), espectadores=('espectadores', 'sum'))
)
taquilla_cine_esp['pelicula_id'] = taquilla_cine_esp['pelicula_id'].astype(int)
taquilla_cine_esp.insert(0, 'taquilla_id', range(1, len(taquilla_cine_esp) + 1))

print(f"\n✓ taquilla_cine_esp: {len(taquilla_cine_esp)} filas (película-año)")
taquilla_cine_esp.head()


Filas de icaa_raw sin pelicula_id tras la unión: 0 / 2812

✓ taquilla_cine_esp: 2810 filas (película-año)


,taquilla_id,pelicula_id,anio,recaudacion,espectadores
0,1,1,2025,42.0,12
1,2,2,2022,83.0,18
2,3,3,2021,129.0,35
3,4,4,2021,237.0,60
4,5,5,2022,1975.0,319


## 9. `subvenciones`

Ya vive en `icaa_peliculas.csv` (`subvenciones_icaa`, lista de conceptos
por película), no hace falta pasar por `icaa_raw`.

In [31]:
def parsear_subvenciones(subvenciones_json_str):
    if pd.isna(subvenciones_json_str) or not subvenciones_json_str:
        return []
    try:
        return json.loads(subvenciones_json_str)
    except Exception:
        return []


registros_subv = []
for _, row in peliculas_completo.iterrows():
    for s in parsear_subvenciones(row['subvenciones_icaa']):
        registros_subv.append({
            'pelicula_id': row['pelicula_id'],
            'concepto': s.get('concepto'),
            'importe': s.get('importe'),
        })

subvenciones = pd.DataFrame(registros_subv)
subvenciones.insert(0, 'subvencion_id', range(1, len(subvenciones) + 1))

print(f"✓ subvenciones: {len(subvenciones)} filas")
subvenciones.head()


✓ subvenciones: 668 filas


,subvencion_id,pelicula_id,concepto,importe
0,1,2,Ayudas Generales para la producción de largome...,280000.0
1,2,8,Ayudas Selectivas para la producción de largom...,50000.0
2,3,10,Ayudas Selectivas para la producción de largom...,140250.0
3,4,16,Ayudas Generales para la producción de largome...,280000.0
4,5,17,Ayudas Selectivas para la producción de largom...,119000.0


## 10. Ajustes de tipo antes de subir a MySQL

`icaa_id`/`tmdb_id`/`person_id_tmdb` son enteros con huecos (`NaN`) --
`Int64` (nullable de pandas) en vez del `int64` normal, para no perder la
condición de entero por culpa de los nulos. `fecha_estreno` se convierte a
fecha real (venía como texto `dd/mm/yyyy`).

In [32]:
for col in ['icaa_id', 'tmdb_id']:
    peliculas[col] = peliculas[col].astype('Int64')
for df_persona in (direccion, guion):
    df_persona['person_id_tmdb'] = df_persona['person_id_tmdb'].astype('Int64')

peliculas['fecha_estreno'] = pd.to_datetime(peliculas['fecha_estreno'], dayfirst=True, errors='coerce')

sin_fecha = peliculas['fecha_estreno'].isna().sum()
print(f"✓ Tipos ajustados. Filas sin fecha_estreno tras la conversión: {sin_fecha} (revisar si es alto)")


✓ Tipos ajustados. Filas sin fecha_estreno tras la conversión: 0 (revisar si es alto)


## 11. Crear tablas en MySQL (con `FOREIGN KEY`)

`pandas.to_sql` nunca crea restricciones reales, así que el DDL se escribe
a mano. Orden de creación: dimensiones antes que las tablas que las
referencian. Al borrar (para poder relanzar el notebook sin duplicar), el
orden es el inverso -- primero lo que tiene FK hacia otras tablas.

In [33]:
engine = create_engine(
    f"mysql+pymysql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}/{os.getenv('DB_NAME')}"
)

TABLAS_EN_ORDEN_DE_DEPENDENCIA = [
    'peliculas', 'distribuidoras', 'generos', 'paises', 'empresas_productoras',
    'direccion', 'guion',
    'peliculas_distribuidoras', 'peliculas_generos', 'peliculas_generos_padre',
    'participacion_pais', 'peliculas_empresas_productoras',
    'peliculas_direccion', 'peliculas_guion',
    'taquilla_cine_esp', 'subvenciones',
]

DDL = """
CREATE TABLE peliculas (
    pelicula_id INT PRIMARY KEY,
    icaa_id INT UNIQUE,
    tmdb_id INT UNIQUE,
    titulo VARCHAR(500),
    titulo_icaa VARCHAR(500),
    fecha_estreno DATE,
    anio INT,
    recaudacion_total DECIMAL(14,2),
    espectadores_total BIGINT,
    subvenciones_total DECIMAL(14,2)
) ENGINE=InnoDB;

CREATE TABLE distribuidoras (
    distribuidora_id INT PRIMARY KEY,
    nombre VARCHAR(500) NOT NULL,
    codigo_icaa_referencia VARCHAR(20)
) ENGINE=InnoDB;

CREATE TABLE generos (
    genero_id INT PRIMARY KEY,
    nombre VARCHAR(100) NOT NULL
) ENGINE=InnoDB;

CREATE TABLE paises (
    pais_id INT PRIMARY KEY,
    nombre VARCHAR(100) NOT NULL,
    es_latam BOOLEAN NOT NULL
) ENGINE=InnoDB;

CREATE TABLE empresas_productoras (
    empresa_id INT PRIMARY KEY,
    nombre VARCHAR(300) NOT NULL
) ENGINE=InnoDB;

CREATE TABLE direccion (
    director_id INT PRIMARY KEY,
    person_id_tmdb INT,
    nombre VARCHAR(300),
    sexo VARCHAR(20),
    fuente_sexo VARCHAR(30),
    pais VARCHAR(10),
    fuente_pais VARCHAR(30),
    nombre_credito_original VARCHAR(300)
) ENGINE=InnoDB;

CREATE TABLE guion (
    guionista_id INT PRIMARY KEY,
    person_id_tmdb INT,
    nombre VARCHAR(300),
    sexo VARCHAR(20),
    fuente_sexo VARCHAR(30),
    pais VARCHAR(10),
    fuente_pais VARCHAR(30),
    nombre_credito_original VARCHAR(300)
) ENGINE=InnoDB;

CREATE TABLE peliculas_distribuidoras (
    pelicula_id INT NOT NULL,
    distribuidora_id INT NOT NULL,
    fuente VARCHAR(20),
    PRIMARY KEY (pelicula_id, distribuidora_id),
    FOREIGN KEY (pelicula_id) REFERENCES peliculas(pelicula_id),
    FOREIGN KEY (distribuidora_id) REFERENCES distribuidoras(distribuidora_id)
) ENGINE=InnoDB;

CREATE TABLE peliculas_generos (
    pelicula_id INT NOT NULL,
    genero_id INT NOT NULL,
    orden INT NOT NULL,
    fuente VARCHAR(20) NOT NULL,
    PRIMARY KEY (pelicula_id, genero_id),
    FOREIGN KEY (pelicula_id) REFERENCES peliculas(pelicula_id),
    FOREIGN KEY (genero_id) REFERENCES generos(genero_id)
) ENGINE=InnoDB;

CREATE TABLE peliculas_generos_padre (
    pelicula_id INT PRIMARY KEY,
    genero_padre VARCHAR(100) NOT NULL,
    FOREIGN KEY (pelicula_id) REFERENCES peliculas(pelicula_id)
) ENGINE=InnoDB;

CREATE TABLE participacion_pais (
    pelicula_id INT NOT NULL,
    pais_id INT NOT NULL,
    porcentaje DECIMAL(6,3),
    porcentaje_fuente VARCHAR(30),
    PRIMARY KEY (pelicula_id, pais_id),
    FOREIGN KEY (pelicula_id) REFERENCES peliculas(pelicula_id),
    FOREIGN KEY (pais_id) REFERENCES paises(pais_id)
) ENGINE=InnoDB;

CREATE TABLE peliculas_empresas_productoras (
    pelicula_id INT NOT NULL,
    empresa_id INT NOT NULL,
    pais_id INT,
    porcentaje_empresa DECIMAL(6,3),
    PRIMARY KEY (pelicula_id, empresa_id),
    FOREIGN KEY (pelicula_id) REFERENCES peliculas(pelicula_id),
    FOREIGN KEY (empresa_id) REFERENCES empresas_productoras(empresa_id),
    FOREIGN KEY (pais_id) REFERENCES paises(pais_id)
) ENGINE=InnoDB;

CREATE TABLE peliculas_direccion (
    pelicula_id INT NOT NULL,
    director_id INT NOT NULL,
    fuente_credito VARCHAR(20),
    PRIMARY KEY (pelicula_id, director_id),
    FOREIGN KEY (pelicula_id) REFERENCES peliculas(pelicula_id),
    FOREIGN KEY (director_id) REFERENCES direccion(director_id)
) ENGINE=InnoDB;

CREATE TABLE peliculas_guion (
    pelicula_id INT NOT NULL,
    guionista_id INT NOT NULL,
    fuente_credito VARCHAR(20),
    PRIMARY KEY (pelicula_id, guionista_id),
    FOREIGN KEY (pelicula_id) REFERENCES peliculas(pelicula_id),
    FOREIGN KEY (guionista_id) REFERENCES guion(guionista_id)
) ENGINE=InnoDB;

CREATE TABLE taquilla_cine_esp (
    taquilla_id INT PRIMARY KEY,
    pelicula_id INT NOT NULL,
    anio INT NOT NULL,
    recaudacion DECIMAL(14,2),
    espectadores BIGINT,
    FOREIGN KEY (pelicula_id) REFERENCES peliculas(pelicula_id)
) ENGINE=InnoDB;

CREATE TABLE subvenciones (
    subvencion_id INT PRIMARY KEY,
    pelicula_id INT NOT NULL,
    concepto VARCHAR(500),
    importe DECIMAL(14,2),
    FOREIGN KEY (pelicula_id) REFERENCES peliculas(pelicula_id)
) ENGINE=InnoDB;
"""

with engine.begin() as conn:
    conn.execute(text("SET FOREIGN_KEY_CHECKS=0"))
    for tabla in reversed(TABLAS_EN_ORDEN_DE_DEPENDENCIA):
        conn.execute(text(f"DROP TABLE IF EXISTS {tabla}"))
    conn.execute(text("SET FOREIGN_KEY_CHECKS=1"))
    for statement in DDL.strip().split(";\n\n"):
        statement = statement.strip()
        if statement:
            conn.execute(text(statement))

print("✓ 16 tablas creadas en MySQL con FOREIGN KEY")


✓ 16 tablas creadas en MySQL con FOREIGN KEY


## 12. Insertar datos

In [34]:
DATAFRAMES = {
    'peliculas': peliculas, 'distribuidoras': distribuidoras, 'generos': generos, 'paises': paises,
    'empresas_productoras': empresas_productoras, 'direccion': direccion, 'guion': guion,
    'peliculas_distribuidoras': peliculas_distribuidoras,
    'peliculas_generos': peliculas_generos, 'peliculas_generos_padre': peliculas_generos_padre,
    'participacion_pais': participacion_pais,
    'peliculas_empresas_productoras': peliculas_empresas_productoras,
    'peliculas_direccion': peliculas_direccion, 'peliculas_guion': peliculas_guion,
    'taquilla_cine_esp': taquilla_cine_esp, 'subvenciones': subvenciones,
}

for tabla in TABLAS_EN_ORDEN_DE_DEPENDENCIA:
    df = DATAFRAMES[tabla]
    df.to_sql(tabla, engine, if_exists='append', index=False)
    print(f"✓ {tabla}: {len(df)} filas insertadas")

engine.dispose()
print("\nMySQL conexión cerrada.")


✓ peliculas: 2052 filas insertadas
✓ distribuidoras: 459 filas insertadas
✓ generos: 69 filas insertadas
✓ paises: 55 filas insertadas
✓ empresas_productoras: 2279 filas insertadas
✓ direccion: 1649 filas insertadas
✓ guion: 2323 filas insertadas
✓ peliculas_distribuidoras: 2165 filas insertadas
✓ peliculas_generos: 1871 filas insertadas
✓ peliculas_generos_padre: 1775 filas insertadas
✓ participacion_pais: 2247 filas insertadas
✓ peliculas_empresas_productoras: 3985 filas insertadas
✓ peliculas_direccion: 2159 filas insertadas
✓ peliculas_guion: 2992 filas insertadas
✓ taquilla_cine_esp: 2810 filas insertadas
✓ subvenciones: 668 filas insertadas

MySQL conexión cerrada.


## 13. Guardar un CSV por tabla

In [35]:
for tabla, df in DATAFRAMES.items():
    ruta = CSV_DIR / f"{tabla}.csv"
    df.to_csv(ruta, index=False, sep=';', quoting=csv.QUOTE_NONNUMERIC)
    print(f"  {ruta} ({ruta.stat().st_size / 1024:.1f} KB)")

print("\n✓ 13 CSV guardados")


  ..\3 - csv\peliculas.csv (181.1 KB)
  ..\3 - csv\distribuidoras.csv (18.5 KB)
  ..\3 - csv\generos.csv (1.4 KB)
  ..\3 - csv\paises.csv (1.2 KB)
  ..\3 - csv\empresas_productoras.csv (89.4 KB)
  ..\3 - csv\direccion.csv (97.5 KB)
  ..\3 - csv\guion.csv (135.6 KB)
  ..\3 - csv\peliculas_distribuidoras.csv (46.2 KB)
  ..\3 - csv\peliculas_generos.csv (45.9 KB)
  ..\3 - csv\peliculas_generos_padre.csv (28.6 KB)
  ..\3 - csv\participacion_pais.csv (74.7 KB)
  ..\3 - csv\peliculas_empresas_productoras.csv (76.6 KB)
  ..\3 - csv\peliculas_direccion.csv (36.3 KB)
  ..\3 - csv\peliculas_guion.csv (56.0 KB)
  ..\3 - csv\taquilla_cine_esp.csv (70.4 KB)
  ..\3 - csv\subvenciones.csv (55.6 KB)

✓ 13 CSV guardados


## 14. Vista `vista_latam`

Para el análisis, en vez de repetir el filtro de participación LATAM en
cada consulta.

In [36]:
engine = create_engine(
    f"mysql+pymysql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}/{os.getenv('DB_NAME')}"
)

VISTA_LATAM_DDL = """
CREATE OR REPLACE VIEW vista_latam AS
SELECT p.*
FROM peliculas p
WHERE EXISTS (
    SELECT 1
    FROM participacion_pais pp
    JOIN paises pa ON pa.pais_id = pp.pais_id
    WHERE pp.pelicula_id = p.pelicula_id
      AND pa.es_latam = TRUE
)
"""

with engine.begin() as conn:
    conn.execute(text(VISTA_LATAM_DDL))
    total_latam = conn.execute(text("SELECT COUNT(*) FROM vista_latam")).scalar()

print(f"✓ vista_latam creada -- {total_latam} películas con participación LATAM")
engine.dispose()


✓ vista_latam creada -- 144 películas con participación LATAM
